# Running SaulLM and Comparing with "Retrieval Conditions"
## Retrieval Conditions: (1) Bare, (2) BM25, and (3) SONAR

Runs SaulLM-7B-Instruct on CaseHOLD with three retrieval conditions (bare / BM25 / SONAR). 

In [2]:
!pip install -q --upgrade pip setuptools wheel
!pip install -q datasets rank-bm25 faiss-cpu requests tqdm pandas matplotlib numpy nltk certifi
#!pip install -q fairseq2 sonar-space
!pip install -q -q fsspec

In [3]:
import subprocess, time, urllib.request

def _ollama_up():
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
        return True
    except Exception:
        return False

if not _ollama_up():
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(30):
        if _ollama_up(): break
        time.sleep(1)
assert _ollama_up(), "ollama server failed to start; run `ollama serve` in a terminal"
print('ollama up')

ollama up


In [4]:
!ollama pull hf.co/mradermacher/Saul-7B-Instruct-v1-GGUF:Q4_K_M
!ollama cp hf.co/mradermacher/Saul-7B-Instruct-v1-GGUF:Q4_K_M saul-7b

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest 
pulling 7034821b79ad: 100% ▕██████████████████▏ 4.4 GB                         
pulling c43332387573: 100% ▕██████████████████▏   67 B                         pulling manifest 
pulling 7034821b79ad: 100% ▕██████████████████▏ 4.4 GB                         
pulling c43332387573: 100% ▕██████████████████▏   67 B                         
pulling 7113dc2fd1dd: 100% ▕██████████████████▏   29 B                         
pulling a40a680f2877: 100% ▕██████████████████▏  552 B                         
verifying sha256 digest 
writing manifest 
success 
]11;?\copied 'hf.co/mradermacher/Saul-7B-Instruct-v1-GGUF:Q4_K_M' to 'saul-7b'


In [5]:
import json, re, time, sys, platform
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import faiss
import matplotlib.pyplot as plt
import datasets
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from sonar.inference_pipelines.text import TextToEmbeddingModelPipeline
from tqdm.auto import tqdm

SEED = 42
N_TEST = 50
K_RETRIEVE = 5
CORPUS_SIZE = 2000
MODELS = ['saul-7b']
RETRIEVER_NAMES = ['bare', 'bm25', 'sonar']
OLLAMA_URL = 'http://localhost:11434/api/chat'
SAVE_JSON = False
JSON_PATH = 'results.json'
RESULTS_DIR = Path('results')
RESULTS_DIR.mkdir(exist_ok=True)

np.random.seed(SEED)
print(f'python={sys.version.split()[0]}  arch={platform.machine()}  datasets={datasets.__version__}')

python=3.12.8  arch=arm64  datasets=4.8.5


In [6]:
ds = load_dataset('casehold/casehold', 'all')
train = ds['train'].select(range(CORPUS_SIZE))
test = ds['test'].select(range(N_TEST))
corpus_texts = [r['citing_prompt'] for r in train]
print(f'corpus: {len(corpus_texts)}  test: {len(test)}')

Using the latest cached version of the dataset since casehold/casehold couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'all' at /Users/nd/.cache/huggingface/datasets/casehold___casehold/all/1.1.0/acc4532a67e0966e9ed7ec4ca543e8532f983b0c (last modified on Wed May 13 20:36:47 2026).


corpus: 2000  test: 50


In [7]:
import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('punkt', quiet=True)
from nltk.tokenize import sent_tokenize
print('nltk ready', flush=True)

bm25 = BM25Okapi([t.lower().split() for t in corpus_texts])

def retrieve_bm25(q, k=K_RETRIEVE):
    scores = bm25.get_scores(q.lower().split())
    return [corpus_texts[i] for i in np.argsort(scores)[-k:][::-1]]

print(f'bm25 ready ({len(corpus_texts)} excerpts)', flush=True)

nltk ready
bm25 ready (2000 excerpts)


In [8]:
corpus_sentences = []
sentence_to_parent = []
for i, text in enumerate(corpus_texts):
    for s in sent_tokenize(text):
        s = s.strip()
        if len(s) > 10:
            corpus_sentences.append(s)
            sentence_to_parent.append(i)
print(f'segmented {len(corpus_texts)} excerpts into {len(corpus_sentences)} sentences', flush=True)

segmented 2000 excerpts into 14889 sentences


In [ ]:
print('loading SONAR encoder...', flush=True)
import time
t0 = time.time()
sonar = TextToEmbeddingModelPipeline(encoder='text_sonar_basic_encoder',
                                     tokenizer='text_sonar_basic_encoder')
print(f'SONAR loaded in {time.time()-t0:.0f}s', flush=True)

parameter load: ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━     0/387   0% -:--:--

In [ ]:
cache = Path(f'sonar_sentences_{CORPUS_SIZE}.npy')
if cache.exists():
    sent_vec = np.load(cache)
    print(f'loaded cached embeddings {sent_vec.shape}', flush=True)
else:
    print(f'encoding {len(corpus_sentences)} sentences in chunks of 500...', flush=True)
    t0 = time.time()
    chunks = []
    for start in range(0, len(corpus_sentences), 500):
        batch = corpus_sentences[start:start+500]
        v = sonar.predict(batch, source_lang='eng_Latn', batch_size=16).numpy().astype('float32')
        chunks.append(v)
        print(f'  {start+len(batch)}/{len(corpus_sentences)} ({time.time()-t0:.0f}s)', flush=True)
    sent_vec = np.concatenate(chunks, axis=0)
    np.save(cache, sent_vec)
    print(f'encoded in {time.time()-t0:.0f}s, saved {cache}', flush=True)

In [ ]:
RETRIEVERS = {'bare': lambda q: None, 'bm25': retrieve_bm25}
print('faiss index + retrievers ready', flush=True)

faiss.normalize_L2(sent_vec)
sonar_idx = faiss.IndexFlatIP(sent_vec.shape[1])
sonar_idx.add(sent_vec)

def retrieve_sonar(q, k=K_RETRIEVE):
    v = sonar.predict([q], source_lang='eng_Latn').numpy().astype('float32')
    faiss.normalize_L2(v)
    _, idx = sonar_idx.search(v, k * 4)
    seen, result = set(), []
    for i in idx[0]:
        parent = sentence_to_parent[i]
        if parent not in seen:
            seen.add(parent)
            result.append(corpus_texts[parent])
            if len(result) == k: break
    return result

RETRIEVERS['sonar'] = retrieve_sonar
print(f'sonar retriever ready (index has {sonar_idx.ntotal} vectors)', flush=True)

In [ ]:
SYSTEM_PROMPT = "You are a legal expert. Read the case excerpt and respond with exactly one letter (A, B, C, D, or E). No explanation. No prose. Just the letter."

def build_prompt(row, retrieved):
    ctx = ''
    if retrieved:
        ctx = 'Similar cases:\n' + '\n\n'.join(f'{i+1}. {d}' for i, d in enumerate(retrieved)) + '\n\n'
    opts = '\n'.join(f'{l}. {row[f"holding_{i}"]}' for i, l in enumerate('ABCDE'))
    return (f'{ctx}Excerpt: {row["citing_prompt"]}\n\nOptions:\n{opts}\n\n'
            f'Answer with a single letter (A, B, C, D, or E).')

def ask_model(prompt, model):
    t0 = time.time()
    try:
        r = requests.post(OLLAMA_URL, json={
            'model': model, 'stream': False,
            'messages': [
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': prompt},
            ],
            'options': {'temperature': 0.0, 'seed': SEED},
        }, timeout=120).json()
        return r['message']['content'].strip(), round(time.time()-t0, 2), None
    except Exception as e:
        return '', round(time.time()-t0, 2), str(e)

def parse_letter(text):
    text = text.strip().upper()
    if text and text[0] in 'ABCDE':
        return 'ABCDE'.index(text[0])
    m = re.search(r'(?:ANSWER|FINAL|CORRECT)\b[^A-E]{0,20}\b([A-E])\b', text)
    if m: return 'ABCDE'.index(m.group(1))
    m = re.findall(r'\b([A-E])\b', text)
    return 'ABCDE'.index(m[-1]) if m else -1

In [ ]:
records = []
for qi, row in enumerate(tqdm(test, desc='questions')):
    q = row['citing_prompt']
    correct = int(row['label'])
    rec = {
        'question_id': qi,
        'excerpt': q,
        'options': {l: row[f'holding_{i}'] for i, l in enumerate('ABCDE')},
        'correct': 'ABCDE'[correct],
        'results': [],
    }
    for model in MODELS:
        for rname in RETRIEVER_NAMES:
            retrieved = RETRIEVERS[rname](q)
            ans, lat, err = ask_model(build_prompt(row, retrieved), model)
            pred = parse_letter(ans) if not err else -1
            rec['results'].append({
                'model': model,
                'retriever': rname,
                'retrieved': retrieved,
                'answer_raw': ans,
                'predicted': 'ABCDE'[pred] if pred >= 0 else None,
                'correct': pred == correct,
                'latency_s': lat,
                'error': err,
            })
    records.append(rec)
print(f'done: {len(records)} questions x {len(MODELS)*len(RETRIEVER_NAMES)} conditions')

In [ ]:
rows = [(r['model'], r['retriever'], r['correct']) for rec in records for r in rec['results']]
df = pd.DataFrame(rows, columns=['model', 'retriever', 'correct'])
acc = df.groupby(['model', 'retriever']).correct.mean().unstack().reindex(columns=RETRIEVER_NAMES)
print(acc.round(3))

fig, ax = plt.subplots(figsize=(7, 4))
acc.plot.bar(ax=ax, width=0.7)
ax.set_ylabel('accuracy')
ax.set_xlabel('model')
ax.set_title(f'CaseHOLD accuracy (N={N_TEST}, corpus={CORPUS_SIZE})')
ax.set_ylim(0, 1)
ax.legend(title='retriever')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

acc.to_csv(RESULTS_DIR / 'table_1_accuracy.csv')
fig.savefig(RESULTS_DIR / 'figure_1_accuracy.png', dpi=200, bbox_inches='tight')
fig.savefig(RESULTS_DIR / 'figure_1_accuracy.pdf', bbox_inches='tight')

def bootstrap_ci(arr, n_iter=1000, alpha=0.05):
    n = len(arr)
    accs = [np.mean(np.random.choice(arr, n, replace=True)) for _ in range(n_iter)]
    return np.percentile(accs, [100*alpha/2, 100*(1-alpha/2)])

print('\n95% bootstrap CI per retriever:')
for r in RETRIEVER_NAMES:
    arr = df[df.retriever == r].correct.values.astype(int)
    lo, hi = bootstrap_ci(arr)
    print(f'  {r:6s}  acc {arr.mean():.3f}  CI [{lo:.3f}, {hi:.3f}]')

lat_rows = [(r['retriever'], r['latency_s']) for rec in records for r in rec['results']]
lat_df = pd.DataFrame(lat_rows, columns=['retriever', 'latency_s'])
print('\nLatency (s) per retriever:')
print(lat_df.groupby('retriever').latency_s.agg(['mean', 'median', 'max']).round(2))

print('\nPublished CaseHOLD reference numbers (for context):')
print('  BERT-large fine-tuned (Zheng 2021):    ~0.70 F1')
print('  LegalBERT fine-tuned (Zheng 2021):     ~0.72 F1')
print('  GPT-4o zero-shot (arXiv 2505.02172):   ~0.74 F1')
print('  AmazonNovaPro (arXiv 2505.02172):      ~0.72 F1')

In [ ]:
def cond_correct(rec, retriever):
    for r in rec['results']:
        if r['retriever'] == retriever:
            return r['correct']

swing = pd.DataFrame([{
    'qid': rec['question_id'],
    'correct': rec['correct'],
    **{r: cond_correct(rec, r) for r in RETRIEVER_NAMES},
} for rec in records])

n = len(swing)
print(f'--- Swing analysis (N={n}) ---')
print(f'sonar correct, bm25 wrong:           {((~swing.bm25) & swing.sonar).sum()}/{n}')
print(f'bm25 correct, sonar wrong:           {(swing.bm25 & (~swing.sonar)).sum()}/{n}')
print(f'any retrieval correct, bare wrong:   {(swing[["bm25","sonar"]].any(axis=1) & (~swing.bare)).sum()}/{n}')
print(f'bare correct, both retrieval wrong:  {(swing.bare & ~swing[["bm25","sonar"]].any(axis=1)).sum()}/{n}')
print(f'all three correct:                   {(swing.bare & swing.bm25 & swing.sonar).sum()}/{n}')
print(f'all three wrong:                     {((~swing.bare) & (~swing.bm25) & (~swing.sonar)).sum()}/{n}')

swing.to_csv(RESULTS_DIR / 'swing_analysis.csv', index=False)
print(f'\nsaved per-question results to {RESULTS_DIR / "swing_analysis.csv"}')

sonar_wins = swing[(~swing.bm25) & swing.sonar].qid.head(3).tolist()
bm25_wins = swing[swing.bm25 & (~swing.sonar)].qid.head(3).tolist()
print(f'\nexample qids where SONAR helped over BM25: {sonar_wins}')
print(f'example qids where BM25 helped over SONAR: {bm25_wins}')

In [ ]:
if SAVE_JSON:
    Path(JSON_PATH).write_text(json.dumps(records, indent=2))
    print(f'wrote {JSON_PATH} ({len(records)} questions, {Path(JSON_PATH).stat().st_size//1024} KB)')